# DeMoSearch — test the complete inference pipeline

Use **existing model checkpoints** to test wall analysis. There are no dataset-preparation, YOLO training or model-training validation steps in this notebook.

| Mode | Input | Pipeline | Result |
| --- | --- | --- | --- |
| 2D | One photo | SAM3 masks → RGB hold types → colors/positions | `scene.json` |
| 3D | Overlapping photos/ZIP or a video | COLMAP wall reconstruction → SAM3 masks → RGB types → multiview mapping | Same JSON schema plus GLB |

All implementation lives in Python modules. This ordinary `.ipynb` is edited directly and only calls those modules through the CLI. The graph algorithm receives the final wall/hold JSON and creates the route separately.

Run the setup and configure input/checkpoint paths. Enable `RUN_PIPELINE` when ready. A CUDA runtime is required for SAM3; 3D additionally needs a CUDA-enabled COLMAP executable.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

ON_COLAB = Path("/content").is_dir()
WORK = Path("/content/demosearch-inference") if ON_COLAB else Path.cwd()
if ON_COLAB:
    PROJECT = WORK / "project"
    WORK.mkdir(parents=True, exist_ok=True)
    if not PROJECT.exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/DiasKzStrong/hold_detection.git", str(PROJECT),
        ], check=True)
else:
    PROJECT = Path.cwd().resolve()
    if not (PROJECT / "src" / "hold_detection").is_dir():
        PROJECT = PROJECT.parent

if not (PROJECT / "src" / "hold_detection").is_dir():
    raise FileNotFoundError("Set PROJECT to the hold_detection repository checkout.")

PYTHON = Path(sys.executable)
if str(PROJECT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT / "src"))

if ON_COLAB:
    from hold_detection.notebook_runtime import prepare_runtime
    PYTHON = prepare_runtime(WORK, PROJECT, sam3=True)

def run_holds(*arguments):
    environment = os.environ.copy()
    environment["PYTHONPATH"] = str(PROJECT / "src") + os.pathsep + environment.get("PYTHONPATH", "")
    command = [str(PYTHON), "-m", "hold_detection.cli", *map(str, arguments)]
    subprocess.run(command, cwd=PROJECT, env=environment, check=True)

print("Project:", PROJECT)
print("Inference Python:", PYTHON)

## Input and ready checkpoints

Set `MODE="2d"` for one photograph; choose `"3d"` for a directory/ZIP of overlapping photos or a video. In Colab, enable Drive mounting if these paths are on Drive.

`TYPE_WEIGHTS` must point to an **already fine-tuned six-class hold detector**, not the generic COCO `yolo26m.pt`. If you deliberately disable `USE_TYPE_MODEL`, SAM3 can still run but grip types require review.

The optional checkpoint-download cell fetches official SAM3 weights only. It does not train a model.

In [ ]:
MOUNT_DRIVE = ON_COLAB
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

MODE = "2d"  # "2d" or "3d"
INPUT_PATH = Path("/content/drive/MyDrive/DeMoSearch/wall.jpg") if ON_COLAB else PROJECT / "data" / "wall.jpg"
SAM_WEIGHTS = PROJECT / "models" / "sam3" / "sam3.pt"
TYPE_WEIGHTS = Path("/content/drive/MyDrive/DeMoSearch/models/hold_types_best.pt") if ON_COLAB else PROJECT / "models" / "hold_types_best.pt"
USE_TYPE_MODEL = True
FETCH_SAM3_CHECKPOINT = False
RUN_PIPELINE = False

RUN_NAME = "photo_demo"
OUTPUT = WORK / "results" / RUN_NAME if ON_COLAB else PROJECT / "runs" / RUN_NAME
COLMAP = "colmap"
MAX_FRAMES = 48
CONFIDENCE = 0.5
SCALE_M_PER_UNIT = None
SCALE_REFERENCE = None
CORRECTIONS = None  # Optional JSON keyed by hold ID; apply after inspecting the first result.

print("Mode:", MODE, "| input:", INPUT_PATH)
print("Output:", OUTPUT)

In [ ]:
if FETCH_SAM3_CHECKPOINT:
    import getpass
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = getpass.getpass("Approved Hugging Face read token (hidden): ")
    run_holds("fetch-sam3", "--output", SAM_WEIGHTS.parent)

print("SAM3 checkpoint exists:", SAM_WEIGHTS.is_file())
print("Hold-type checkpoint exists:", TYPE_WEIGHTS.is_file())
if RUN_PIPELINE:
    if MODE not in {"2d", "3d"}:
        raise ValueError("MODE must be 2d or 3d.")
    if not INPUT_PATH.exists():
        raise FileNotFoundError(INPUT_PATH)
    if not SAM_WEIGHTS.is_file():
        raise FileNotFoundError("Set SAM_WEIGHTS or enable FETCH_SAM3_CHECKPOINT.")
    if USE_TYPE_MODEL and not TYPE_WEIGHTS.is_file():
        raise FileNotFoundError("Set TYPE_WEIGHTS to an existing hold checkpoint or explicitly disable USE_TYPE_MODEL.")

## 1. Prepare and preview the capture

Images are oriented and bounded to 1600px. Video is sampled into up to `MAX_FRAMES` views. The 2D mode accepts exactly one photo. For 3D, use a sharp capture with parallax and substantial overlap.

Completed preparation can be reused when rerunning the same input/settings. Choose a new output directory after changing input or model settings.

In [ ]:
CAPTURE = OUTPUT / "capture"
if RUN_PIPELINE:
    run_holds("prepare-capture", INPUT_PATH, "--mode", MODE, "--output", CAPTURE,
              "--max-frames", MAX_FRAMES)

capture_path = CAPTURE / "capture.json"
if capture_path.is_file():
    manifest = json.loads(capture_path.read_text())
    print("Prepared frames:", len(manifest["frames"]))
    from IPython.display import display
    from PIL import Image
    for frame in manifest["frames"][:3]:
        with Image.open(CAPTURE / frame["path"]) as image:
            preview = image.copy()
            preview.thumbnail((600, 450))
            display(preview)
else:
    print("Set RUN_PIPELINE=True to prepare the selected input.")

## 2. Reconstruct the wall section — 3D only

COLMAP estimates cameras, stereo depth and a mesh. The pipeline exports GLB with trimesh. SAM3 will use the **undistorted registered images**, so its masks agree with the camera/depth coordinates.

COLMAP must already be installed with CUDA support inside the selected environment/container. Installing `pycolmap` alone does not provide this CLI workflow.

Without calibration, reconstruction coordinates have arbitrary units and meter-valued JSON fields stay `null`. Set a known `SCALE_M_PER_UNIT`, or supply a scale-reference JSON containing two pixels in an **undistorted** image and their known distance. Scale does not establish gravity orientation.

In [ ]:
RECONSTRUCTION = OUTPUT / "reconstruction" / "reconstruction.json" if MODE == "3d" else None
if RUN_PIPELINE and MODE == "3d":
    arguments = ["reconstruct-wall", CAPTURE, "--output", OUTPUT / "reconstruction", "--colmap", COLMAP]
    if SCALE_M_PER_UNIT is not None:
        arguments += ["--scale-m-per-unit", SCALE_M_PER_UNIT]
    if SCALE_REFERENCE is not None:
        arguments += ["--scale-reference", SCALE_REFERENCE]
    run_holds(*arguments)

if RECONSTRUCTION is not None and RECONSTRUCTION.is_file():
    reconstruction = json.loads(RECONSTRUCTION.read_text())
    print("Registered images:", reconstruction["registered_images"], "/", reconstruction["capture_images"])
    print("Components:", reconstruction["component_count"], "| units:", reconstruction["units"])
    print("GLB:", reconstruction["glb_path"])
elif MODE == "2d":
    print("2D mode: reconstruction is skipped.")

## 3. Segment holds and infer RGB types

SAM3 provides the instance masks. The ready hold-type detector supplies six-class RGB proposals through one-to-one overlap matching. No fitting or weight updates occur here.

Each frame runs in a fresh SAM3 process. Completed frame batches are stored and reused on resume. Inspect a failed frame's log under `OUTPUT/logs` if a GPU process exits.

In [ ]:
if RUN_PIPELINE:
    arguments = ["segment-capture", CAPTURE, "--run", OUTPUT,
                 "--sam-weights", SAM_WEIGHTS, "--device", "0", "--confidence", CONFIDENCE]
    if USE_TYPE_MODEL:
        arguments += ["--type-weights", TYPE_WEIGHTS]
    if RECONSTRUCTION is not None:
        arguments += ["--reconstruction-path", RECONSTRUCTION]
    run_holds(*arguments)
else:
    print("Inference disabled until RUN_PIPELINE=True.")

## 4. Fuse observations and export the common scene

2D exports frame-local positions/sizes. 3D lifts mask pixels through stereo depth, combines observations of the same Physical Hold and attaches reconstructed-mesh evidence. Single-view and unmapped candidates remain available for review.

Both modes export the same hold-field schema. Unknown types need manual confirmation or an explicit policy in the teammate's algorithm. A partial scan cannot measure hidden grip area, lip depth or true hold volume; unavailable values remain null.

In [ ]:
if RUN_PIPELINE:
    arguments = ["export-wall", OUTPUT, "--mode", MODE]
    if RECONSTRUCTION is not None:
        arguments += ["--reconstruction-path", RECONSTRUCTION]
    if CORRECTIONS is not None:
        arguments += ["--annotations-path", CORRECTIONS]
    run_holds(*arguments)

SCENE_PATH = OUTPUT / "scene.json"
if SCENE_PATH.is_file():
    scene = json.loads(SCENE_PATH.read_text())
    print("Mode:", scene["analysis_mode"], "| holds:", len(scene["holds"]))
    print("Units:", scene["wall"]["units"])
    print("Review:", scene["review"])
    print("Input for the graph algorithm:", SCENE_PATH)
    print("Review gallery:", OUTPUT / "inference" / "index.html")
    from IPython.display import display
    import pandas as pd
    display(pd.DataFrame(scene["holds"]).head(30))
else:
    scene = None
    print("No scene.json yet.")

In [ ]:
if scene and scene["holds"]:
    import matplotlib.pyplot as plt
    positions = [hold["position_2d_normalized"] for hold in scene["holds"]]
    if MODE == "2d":
        fig, ax = plt.subplots(figsize=(6, 7))
        ax.scatter([p[0] for p in positions], [p[1] for p in positions])
        ax.set(xlabel="Normalized x", ylabel="Normalized y", xlim=(0, 1), ylim=(1, 0),
               title="Detected hold centers")
        plt.show()
    else:
        mapped = [hold for hold in scene["holds"] if hold["position_3d"] is not None]
        if mapped:
            points = [hold["position_3d"] for hold in mapped]
            fig = plt.figure(figsize=(8, 6))
            ax = fig.add_subplot(projection="3d")
            ax.scatter([p[0] for p in points], [p[1] for p in points], [p[2] for p in points])
            ax.set(xlabel="Reconstruction x", ylabel="Reconstruction y", zlabel="Reconstruction z",
                   title="Mapped Physical Holds (reconstruction units)")
            plt.show()

## 5. Optional matching to annotated GLB

For blue V5 / yellow V3, use an explicit 4×4 transform from our reconstruction units to the annotated GLB's meter frame. Independent COLMAP and Polycam captures do not align automatically. The matcher assigns named hold objects one-to-one within a distance gate and retains unmatched holds.

In [ ]:
MATCH_ANNOTATED = False
ANNOTATED_GLB = PROJECT / "data" / "wallC_blueV5_marked.glb"
SCENE_TO_ANNOTATED = PROJECT / "data" / "scene_to_annotated.json"
if MATCH_ANNOTATED:
    run_holds("match-annotated-glb", SCENE_PATH, "--annotated-glb", ANNOTATED_GLB,
              "--transform", SCENE_TO_ANNOTATED, "--output", OUTPUT / "scene_matched.json")

## The same inference workflow on HPC

The implementation also runs without Jupyter. Configure the environment and ready checkpoint paths, then submit the analysis chain:

```bash
export DEMO_PROJECT=/scratch/NetID/hold_detection
export DEMO_PYTHON=/scratch/NetID/runtime/bin/python
export DEMO_SOURCE=/scratch/NetID/captures/wall.mov
export DEMO_RUN=/scratch/NetID/hold-runs/wallC
export DEMO_MODE=3d
export DEMO_SAM_WEIGHTS=/scratch/NetID/models/sam3/sam3.pt
export DEMO_TYPE_WEIGHTS=/scratch/NetID/models/hold_types_best.pt
export DEMO_COLMAP=/path/to/cuda-colmap/bin/colmap
bash scripts/slurm/submit_wall.sh
```

The chain is prepare → reconstruct → segment → export, with `afterok` dependencies. For a single photo use `DEMO_MODE=2d`; reconstruction is skipped. The alternative single-job entry point is `DEMO_STAGE=analyze sbatch scripts/slurm/wall-gpu.sbatch`.

These are text examples and are not submitted automatically. Model fitting remains in separate Python/SLURM jobs outside this notebook.

Review missing/extra masks, ambiguous types, registered-view count, repeated/unmapped holds, units and spatial estimates before giving the scene to the graph algorithm. Synthetic tests do not establish real wall accuracy. See [the complete pipeline guide](../docs/wall-pipeline.md).